In [1]:
import sklearn.metrics
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
import glob

experiment_folders = glob.glob(f"predictions/*")

experiments = ["cavanagh", "dieleman", "resnet50", "newsimple"]

for model in experiments:
    print(f"{model}")
    for experiment_folder in experiment_folders:
        print(experiment_folder.split("/")[-1])
        accuracies = []
        tprs = []
        tnrs = []
        f1s = []
        recalls = []
        for fold_path in (glob.glob(f"{experiment_folder}/128x128/{model}/*.csv")):
            # print(fold_path)
            if os.path.exists(fold_path):
                data = pd.read_csv(fold_path)

                true_labels = data['True'].to_numpy()
                predicted_labels = (data['Prediction'].to_numpy() >= 0.5).astype(int)


                accuracy = sklearn.metrics.accuracy_score(true_labels, predicted_labels)
                accuracies.append(accuracy * 100)
                # print("Acc: ", f"{accuracy:.4f}")

                matrix = sklearn.metrics.confusion_matrix(true_labels, predicted_labels)
                tnr = matrix[0,0] / (matrix[0,0] + matrix[1, 0])
                tpr = matrix[1,1] / (matrix[1,1] + matrix[0, 1])
                # print("TPR: ", f"{tpr:.4f}")
                # print("TNR: ", f"{tnr:.4f}")

                tprs.append(tpr * 100)
                tnrs.append(tnr * 100)

                recalls.append(sklearn.metrics.recall_score(true_labels, predicted_labels) * 100)

                f1 = sklearn.metrics.f1_score(true_labels, predicted_labels) * 100
                f1s.append(f1)
            else:
                # raise ValueError()
                pass
        if(accuracies):
            print(f"Acc: {np.nanmean(accuracies):.2f}", f"std:{np.nanstd(accuracies):.2f}")
            print(f"TPR: {np.nanmean(tprs):.2f}", f"std:{np.nanstd(tprs):.2f}")
            print(f"Rec: {np.nanmean(recalls):.2f}", f"std:{np.nanstd(recalls):.2f}")
            print(f"TNR: {np.nanmean(tnrs):.2f}", f"std:{np.nanstd(tnrs):.2f}")
            print(f"F1: {np.nanmean(f1s):.2f}", f"std:{np.nanstd(f1s):.2f}")
            print("-----")

cavanagh
predictions\baseline_no_augmentations
Acc: 94.71 std:0.12
TPR: 89.80 std:1.18
Rec: 92.19 std:1.47
TNR: 96.80 std:0.56
F1: 90.96 std:0.23
-----
predictions\basic_augmentations
Acc: 95.36 std:0.14
TPR: 90.73 std:0.68
Rec: 93.49 std:0.95
TNR: 97.33 std:0.37
F1: 92.09 std:0.26
-----
predictions\basic_augmentations_and_random_noise
Acc: 95.34 std:0.10
TPR: 90.73 std:0.55
Rec: 93.42 std:0.54
TNR: 97.30 std:0.21
F1: 92.05 std:0.16
-----
predictions\basic_augmentations_center_noise
Acc: 95.29 std:0.19
TPR: 90.67 std:0.51
Rec: 93.27 std:0.47
TNR: 97.24 std:0.19
F1: 91.95 std:0.32
-----
predictions\basic_augmentations_outside_noise
Acc: 95.39 std:0.17
TPR: 90.86 std:0.60
Rec: 93.43 std:0.75
TNR: 97.30 std:0.29
F1: 92.12 std:0.30
-----
dieleman
predictions\baseline_no_augmentations
Acc: 94.70 std:0.12
TPR: 89.68 std:0.79
Rec: 92.28 std:0.93
TNR: 96.83 std:0.36
F1: 90.95 std:0.21
-----
predictions\basic_augmentations
Acc: 95.42 std:0.17
TPR: 91.18 std:0.81
Rec: 93.17 std:0.72
TNR: 97.20 s

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import iqr

dr7_full = pd.read_csv("../SDSS_Images/final_dataset_classes.csv")

dr7 = pd.read_csv("Redshifts_DR7.csv")
dr17 = pd.read_csv("Redshifts_DR17.csv")

merged_with_dr7 = pd.merge(dr7_full, dr7[["Id", "z"]], on="Id", how = "left")
merged = pd.merge(merged_with_dr7, dr17[["Id", "z"]], on="Id", how = "left")
merged["z"] = np.where(np.isnan(merged['z_x']), merged["z_y"], merged["z_x"])
merged = merged.drop(columns = ["z_x", "z_y"])

print(len(dr7_full))
iqr_z = iqr(merged[merged["z"].between(0, 0.5)]["z"], nan_policy="omit")
q1, q3 = np.nanpercentile(merged[merged["z"].between(0, 0.5)]["z"], (25, 75), )
q1, q3

print(q1 - 1.5 * iqr_z, q3 + 1.5 * iqr_z)

# merged[merged["z"].between(0.25, 0.5)]
merged["z"].mean() + 3 * merged["z"].std()

/opt/homebrew/Caskroom/miniforge/base/envs/data_analysis_env/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3203: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


315942
-0.03504575000000004 0.23142825000000006


0.28210013615017915